In [1]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
import time

from scipy.signal import resample_poly

WINDOWING SAMPLERATE AND OVERLAP

In [2]:
FS = 200                     # Sampling frequency (Hz)
WINDOW_SEC = 30              # Sliding window length (seconds)
WINDOW_SAMPLES = FS * WINDOW_SEC
STEP_SEC = 1                 # Step size for 1 Hz output


 Resample signal using polyphase filtering

In [3]:
def resample_signal(signal, fs_in=256, fs_out=200):

    return resample_poly(signal, fs_out, fs_in)

In [4]:
import warnings
warnings.filterwarnings("ignore")


FEATURE EXTRACTION VER 1

In [5]:
def extract_features_from_full_gsr_ppg(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # Resample signals to 200 Hz

    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI )
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]
    rr_intervals = np.diff(rpeaks) / FS * 1000
    rr_times = rpeaks[1:]

    features = []
    step_samples = FS * STEP_SEC

    for start in range(0, len(df) - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # HR
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # HRV
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)
        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

        # EDA
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
        window["SCR_Count"] = int(eda_window["SCR_Peaks"].sum())

        # Metadata
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION WITH RR VALUES FOR ANALYSIS and SCR_RATE

In [8]:
def extract_features_with_rr(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]

    rr_intervals = np.diff(rpeaks) / FS * 1000 # ms

    # bättre tidsrepresentation (mitten av intervallet)
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(ppg)  # 🔥 FIX: använd resamplad signal

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # HRV + RR diagnostics
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # filtrera orimliga RR
        rr_win = rr_win[(rr_win > 300) & (rr_win < 1500)]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)

            # filtrera extrema diff
            diff_rr = diff_rr[np.abs(diff_rr) < 200]

            # HRV
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics 
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_diff_std"] = np.std(diff_rr)

            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan

            window["RR_valid"] = 0

        # -------------------------
        # EDA
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
       
        scr_count = int(eda_window["SCR_Peaks"].sum())

        window["SCR_Rate"] = scr_count/ WINDOW_SEC

        

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

In [12]:
def extract_features_with_PPG_foucus(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Remove edge artifacts
    # -------------------------
    ppg = ppg[FS*2:-FS*2]
    eda = eda[FS*2:-FS*2]

    # -------------------------
    # Process signals (bättre peak detection)
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(
        ppg,
        sampling_rate=FS,
        method="elgendi",   
        method_peaks="charlton"   # bättre än default
    )

    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    signal_clean = signals_ppg["PPG_Clean"].values
    rpeaks = info_ppg["PPG_Peaks"]

    # -------------------------
    # Peak refinement (KRITISK)
    # -------------------------
    def refine_peaks(signal, peaks, fs, window=0.05):
        refined = []
        w = int(window * fs)

        for p in peaks:
            start = max(0, p - w)
            end = min(len(signal), p + w)

            segment = signal[start:end]
            new_p = start + np.argmax(segment)

            refined.append(new_p)

        return np.array(refined)

    rpeaks = refine_peaks(signal_clean, rpeaks, FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rr_intervals = np.diff(rpeaks) / FS * 1000  # ms

    # bättre tidsrepresentation (mitten av intervallet)
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(signal_clean)  # viktigt

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # HRV + RR diagnostics
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # -------------------------
        # Strikt RR filtering
        # -------------------------
        rr_win = rr_win[(rr_win > 400) & (rr_win < 1200)]

        if len(rr_win) >= 10:

            diff_rr = np.diff(rr_win)

            # bättre diff-filter
            valid_diff = np.abs(diff_rr) < 150
            diff_rr = diff_rr[valid_diff]

            # quality metric
            rr_quality = np.sum(valid_diff) / len(valid_diff)

            # -------------------------
            # HRV
            # -------------------------
            if len(diff_rr) > 0:
                window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
                window["RR_diff_std"] = np.std(diff_rr)
            else:
                window["HRV_RMSSD"] = np.nan
                window["RR_diff_std"] = np.nan

            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics 
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_quality"] = rr_quality
            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan
            window["RR_quality"] = 0
            window["RR_valid"] = 0

        # -------------------------
        # EDA 
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()

        scr_count = int(eda_window["SCR_Peaks"].sum())
        window["SCR_Rate"] = scr_count / WINDOW_SEC

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

In [19]:
def extract_features_with_PPG_foucus_centroid(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Remove edge artifacts
    # -------------------------
    ppg = ppg[FS*2:-FS*2]
    eda = eda[FS*2:-FS*2]

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(
        ppg,
        sampling_rate=FS,
        method="elgendi",
        method_peaks="charlton"
    )

    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    signal_clean = signals_ppg["PPG_Clean"].values
    rpeaks = info_ppg["PPG_Peaks"]

    # -------------------------
    # 🔥 Centroid refinement
    # -------------------------
    def refine_peaks_centroid(signal, peaks, fs, window=0.05):
        refined = []
        w = int(window * fs)

        for p in peaks:
            start = max(0, p - w)
            end = min(len(signal), p + w)

            segment = signal[start:end]
            x = np.arange(start, end)

            # gör signal positiv
            y = segment - np.min(segment)

            # fallback om något går fel
            if np.sum(y) == 0 or len(segment) == 0:
                refined.append(p)
                continue

            centroid = np.sum(x * y) / np.sum(y)

            refined.append(int(centroid))

        return np.array(refined)

    # 🔥 använd centroid istället för max
    rpeaks = refine_peaks_centroid(signal_clean, rpeaks, FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rr_intervals = np.diff(rpeaks) / FS * 1000  # ms

    # bättre tidsrepresentation
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(signal_clean)

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # RR selection
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # -------------------------
        # RR filtering
        # -------------------------
        rr_win = rr_win[(rr_win > 400) & (rr_win < 1200)]

        if len(rr_win) >= 10:

            diff_rr = np.diff(rr_win)

            valid_diff = np.abs(diff_rr) < 150
            diff_rr = diff_rr[valid_diff]

            rr_quality = np.sum(valid_diff) / len(valid_diff)

            # -------------------------
            # HRV
            # -------------------------
            if len(diff_rr) > 0:
                window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
                window["RR_diff_std"] = np.std(diff_rr)
            else:
                window["HRV_RMSSD"] = np.nan
                window["RR_diff_std"] = np.nan

            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_quality"] = rr_quality
            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan
            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan
            window["RR_quality"] = 0
            window["RR_valid"] = 0

        # -------------------------
        # EDA
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()

        scr_count = int(eda_window["SCR_Peaks"].sum())
        window["SCR_Rate"] = scr_count / WINDOW_SEC

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION FOR TRAIN PART 1-48 FULL FILE (full_gsr_ppg.csv)

In [20]:
DATA_DIR = Path("../data/raw/Participants")

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)
print("Found files:", len(participant_files))
for f in participant_files[:5]:
    print(f)
# -------------------------
# Select participants 1–48
# -------------------------
participant_files = participant_files[:48]

all_features = []

start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):
    participant_id = file.parent.name

    print(f"\nProcessing {participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    participant_features = extract_features_with_PPG_foucus_centroid(
        df, participant_id
    )

    all_features.append(participant_features)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    print(
        f"Elapsed: {elapsed/60:.1f} min | "
        f"Remaining: {remaining/60:.1f} min"
    )


Found files: 60
..\data\raw\Participants\Part1\full_gsr_ppg.csv
..\data\raw\Participants\Part2\full_gsr_ppg.csv
..\data\raw\Participants\Part3\full_gsr_ppg.csv
..\data\raw\Participants\Part4\full_gsr_ppg.csv
..\data\raw\Participants\Part5\full_gsr_ppg.csv


Processing participants:   0%|          | 0/48 [00:00<?, ?it/s]


Processing Part1 (1/48)


Processing participants:   2%|▏         | 1/48 [02:08<1:40:57, 128.89s/it]

Elapsed: 2.1 min | Remaining: 101.0 min

Processing Part2 (2/48)


...

Elapsed: 41.5 min | Remaining: 1.8 min

Processing Part47 (47/48)


Processing participants:  98%|█████████▊| 47/48 [42:36<00:55, 55.47s/it]

Elapsed: 42.6 min | Remaining: 0.9 min

Processing Part48 (48/48)


Processing participants: 100%|██████████| 48/48 [43:50<00:00, 54.80s/it]

Elapsed: 43.8 min | Remaining: 0.0 min


DF CHECK BEFORE LOG AND CLIPPING 

In [21]:
features_df = pd.concat(all_features, ignore_index=True)
features_df = features_df.dropna().reset_index(drop=True)
features_df.describe()


,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,EDA_Tonic,EDA_Phasic,SCR_Rate,time_sec
count,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.000000,108236.0,108236.000000,108236.000000,108236.000000,108236.000000
mean,79.562216,48.450728,48.139089,59.858577,773.029877,59.063043,643.852369,913.745196,39.366191,0.944811,1.0,2628.011100,0.015533,0.052994,1130.658376
std,11.762024,17.194039,16.938379,25.959638,106.700787,25.587180,111.017178,130.721666,5.579251,0.089690,0.0,11669.142072,79.379133,0.057530,656.039790
min,52.600662,8.021961,8.017150,10.176887,427.734375,10.083945,405.000000,460.000000,21.000000,0.314286,1.0,-699.751593,-3659.799728,0.000000,0.000000
25%,71.387263,35.191782,35.101920,40.939412,698.837209,40.416072,565.000000,825.000000,36.000000,0.923077,1.0,84.732295,-0.123579,0.000000,563.000000
50%,78.188259,48.410542,48.206047,55.501708,769.487179,54.762246,650.000000,915.000000,39.000000,1.000000,1.0,171.323321,-0.000132,0.033333,1127.000000
75%,86.338322,61.550642,61.048261,74.428992,841.250000,73.410045,715.000000,1005.000000,43.000000,1.000000,1.0,368.137855,0.124958,0.100000,1696.000000
max,167.366956,108.512672,108.397417,209.321595,1129.565217,206.125658,1005.000000,1195.000000,67.000000,1.000000,1.0,92911.198628,4880.031171,0.366667,2305.000000


LOG AND CLIPPING FEATURES 

In [22]:

# EDA stabilization, log-transform and clip extreme values

EPS = 1e-6

features_df["EDA_Tonic_log"] = np.log(features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS)
features_df["EDA_Phasic_log"] = np.log(np.abs(features_df["EDA_Phasic"]) + EPS)

# SCR LOG
features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])

#Clip extreme values

features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0,200)
features_df["HRV_SDNN"] = features_df["HRV_SDNN"].clip(0,200)


# EDA CLIPP
features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

# SCR CLIPP

features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)


SAVE FULL FILE FOR ANALYS

In [23]:
features_df.to_csv("../data/features_extraction_full_file_analys_RR_centroid.csv", index=False)

CREATE DF FOR SOM MODEL WITH CORRECT FEATURES

In [ ]:
# Feature columns used for SOM
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Rate",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

# SOM input matrix
X_features = features_df[FEATURE_COLS]

# (Optional) metadata kept separately
metadata = features_df[["participant", "time_sec"]]

X_features.describe()

In [ ]:
print(features_df["SCR_Rate"].describe())

In [24]:
features_df = features_df[features_df["RR_quality"] > 0.7]

# Feature columns used for SOM
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
]

# SOM input
X_features_PPG = features_df[FEATURE_COLS]

# metadata
metadata = features_df[["participant", "time_sec"]]

# check
X_features_PPG.describe()

,HR,HRV_RMSSD,HRV_SDNN
count,104976.000000,104976.000000,104976.000000
mean,79.505632,47.735581,57.832439
std,11.658813,16.844031,23.335480
min,52.600662,8.021961,10.176887
25%,71.358806,34.764013,40.440871
50%,78.205153,47.673831,54.540574
75%,86.366197,60.566768,72.229098
max,155.145444,100.415802,170.593132


SAVE FULL FILE FOR SOM MODEL

In [25]:
X_features_PPG.to_csv(    
    "../data/features_full_file_SOM_PPG_centroid.csv",
    index=False
)


FEATURE EXTRACTION 49-60 FOR TEST with clipping and log

FULL FILE (full_gsr_ppg.csv)

SAVED IN /test_features_windows --> 1: all participants 2: participant_49..60

In [28]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("test_features_windows_ppg_centroid")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)

print("Total files found:", len(participant_files))

# -------------------------
# Select participants 49–60
# -------------------------
participant_files = participant_files[48:60]

print("Selected participants:")
for f in participant_files:
    print(f.parent.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):

    participant_id = int(file.parent.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION WITH RR
    # -------------------------
    features_df = extract_features_with_PPG_foucus_centroid(df, participant_id)
    features_df = features_df[
    (features_df["RR_quality"] > 0.7)
    ]

    # -------------------------
    # EDA STABILIZATION 
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING 
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

   
    # SCR LOG
    features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])
    features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)

    

    # EDA clipping 
    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA 
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_windows_features_PPG_centroid.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "features_49_60_ALL_WINDOWS_PPG_centroid.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)

Total files found: 60
Selected participants:
Part49
Part50
Part51
Part52
Part53
Part54
Part55
Part56
Part57
Part58
Part59
Part60


Processing participants:   0%|          | 0/12 [00:00<?, ?it/s]

Processing Part49 (1/12)


Processing participants:   8%|▊         | 1/12 [00:51<09:23, 51.21s/it]

Elapsed: 0.9 min | Remaining: 9.4 min
Processing Part50 (2/12)


...

Elapsed: 9.5 min | Remaining: 1.9 min
Processing Part59 (11/12)


Processing participants:  92%|█████████▏| 11/12 [10:32<00:59, 59.94s/it]

Elapsed: 10.5 min | Remaining: 1.0 min
Processing Part60 (12/12)


Processing participants: 100%|██████████| 12/12 [11:52<00:00, 59.40s/it]


Elapsed: 11.9 min | Remaining: 0.0 min

✅ DONE
(27106, 16)


FEATURE EXTRACTION BY_BLOCK 

FILE PATH -->  file = p_dir / "by_block" / "7_gsr_ppg_.csv" (CHANGE FOR DIFFERENT BLOCK)

SAVE BOTH ALL PARTICIPANT AND ONE PER PARTICIPANT 

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("test_features_IQTEST_8")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect stroop files
# -------------------------
participant_dirs = sorted(
    [p for p in DATA_DIR.glob("Part*")],
    key=lambda x: int(x.name.replace("Part", ""))
)

# Select 49–60
participant_dirs = participant_dirs[48:60]

print("Selected participants:")
for p in participant_dirs:
    print(p.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, p_dir in enumerate(tqdm(participant_dirs, desc="Processing participants")):

    participant_id = int(p_dir.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_dirs)})")

    # -------------------------
    # FILE PATH (stroop)
    # -------------------------
    file = p_dir / "by_block" / "8_gsr_ppg_IQtest.csv"

    if not file.exists():
        tqdm.write(f"⚠️ Missing file for Part{participant_id}")
        continue

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION
    # -------------------------
    features_df = extract_features_with_rr(df, participant_id)

    # -------------------------
    # EDA STABILIZATION
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

    # SCR LOG
    features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])
    features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)

    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_IQTEST_8.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_dirs) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "IQTEST_8_ALLPARTICIPANTS_WINDOWS.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)

Selected participants:
Part49
Part50
Part51
Part52
Part53
Part54
Part55
Part56
Part57
Part58
Part59
Part60


Processing participants:   0%|          | 0/12 [00:00<?, ?it/s]

Processing Part49 (1/12)


Processing participants:   8%|▊         | 1/12 [00:02<00:25,  2.30s/it]

Elapsed: 0.0 min | Remaining: 0.4 min
Processing Part50 (2/12)


...

Elapsed: 0.5 min | Remaining: 0.1 min
Processing Part59 (11/12)


Processing participants:  92%|█████████▏| 11/12 [00:31<00:03,  3.20s/it]

Elapsed: 0.5 min | Remaining: 0.0 min
Processing Part60 (12/12)


Processing participants: 100%|██████████| 12/12 [00:34<00:00,  2.91s/it]

Elapsed: 0.6 min | Remaining: 0.0 min

✅ DONE
(3253, 15)
